# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Reading `docs/flyrank-seo-research-march-2026.pdf` -- FlyRank's own March 2026 report on 341,701 content pieces across 57 brands. Two CONFIRMED findings, with constructive methodology questions on each.

---

### Finding #1 -- "The Anatomy of Growing Content" (CONFIRMED)

**The claim:** comparing pages trending upward (74.8K, 30d-vs-prev-30d impressions) against pages trending downward (45.6K), growing pages are reported as ~38% longer (3,180 vs 2,311 avg words) and ~20% younger (184 vs 230 days), with slightly better average position.

**My methodology question, asked constructively:** the two structural signals (length, age) are reported as separate bivariate group comparisons, not jointly in a single model -- so it's not clear whether length and age are independently associated with growth, or whether one is doing most of the work and the other is riding along because growing pages also happen to be newer (longer pages might just be more recently *and* thoroughly authored, making age and length correlated with each other, not just with growth). A stronger version of this finding would report a joint model (e.g. logistic regression with both age and length as covariates) the way my own w05 notebook does, to see whether each signal survives holding the other constant.

**A second, more personal question:** my own w04 signal audit found the *opposite* direction for length on a different label -- in my March 2026 slice, longer content associates with a *lower* rate of reaching page-one (a static state), not a higher growth rate (a momentum label). These aren't necessarily contradictory -- a page can be actively growing from a low base while still not yet being page-one -- but it's a genuine open question whether "longer helps momentum" and "longer hurts static position" can both be true at once, and I don't yet have the data to settle it. I'm noting this tension rather than picking a side.

---

### Finding #2 -- "The Content Performance Curve" (CONFIRMED)

**The claim:** Health Score by content-age bucket shows a rise to a peak at 61-90 days (score ~33), a "decay cliff" by 271-365 days (score drops to ~14), and a rebound at 365+ days (score ~25) attributed to pages that were refreshed.

**My methodology question #1:** Health Score is a composite of four different metrics (impressions, position, CTR, scroll depth) combined into one number. A composite can produce a curve shape that no single underlying metric actually has -- e.g. the "365+ rebound" could be entirely a scroll-depth or CTR effect with position flat, or vice versa. The paper doesn't show the age curve decomposed by each raw component separately, so I can't tell which underlying signal is actually driving the lifecycle shape versus which are just along for the ride in the composite.

**My methodology question #2 -- survivorship, asked constructively:** the paper notes that some of its cuts use an "active-content" feature-vector restricted to pages with nonzero impressions and sessions in the last 90 days. If a similar filter (implicit or explicit) applies to the age-curve sample, the 365+ bucket would only contain pages that are *still being tracked with real traffic* -- meaning old, abandoned, zero-traffic pages would already be excluded before the curve is drawn. That would mean the "rebound" isn't really "old pages recover when refreshed" so much as "the old pages that survived long enough to still have traffic tend to have been refreshed" -- a subtly different and much narrower claim. The paper's own text is actually careful here ("this is not evidence that age naturally reverses decline on its own") -- my question is really whether the sample itself already silently excludes the pages that would show the alternative story.

In [1]:
# Supporting check -- confirm the paper file is present and note its verified scale for citation.
import os
print("Paper present:", os.path.exists("../../docs/flyrank-seo-research-march-2026.pdf"))
print("Reported scale: 341,701 content pieces, 57 brands, 469.9M impressions, 1.51M clicks (per the paper's Study Scope section)")
print("Finding #1 sample sizes: 74.8K growing vs 45.6K declining -- both comfortably above the 50-row floor")


Paper present: True
Reported scale: 341,701 content pieces, 57 brands, 469.9M impressions, 1.51M clicks (per the paper's Study Scope section)
Finding #1 sample sizes: 74.8K growing vs 45.6K declining -- both comfortably above the 50-row floor


## 2. My model under an honest split (before/after)

Re-running my w05 Logistic Regression under an **ungrounded row-level random split** ("before") vs the **grouped-by-client split** ("after") I actually used -- same data, same features, same model, only the split changes. The gap between them is itself the finding: it shows how much of the model's apparent skill would have been an illusion from memorizing client-specific quirks.

In [2]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTH   = "2026-03"

page_agg = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions_win,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

content = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume, competition, backlinks, content_type, main_intent,
           GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

data = page_agg.merge(content, on="content_hash_id", how="left").dropna(
    subset=["word_count", "search_volume", "competition", "backlinks"]
)
print(f"modeling frame: {len(data):,} pages | base rate: {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

modeling frame: 53,892 pages | base rate: 56.2%


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

NUM_FEATURES = ["word_count", "content_age_days", "search_volume", "competition", "backlinks"]
CAT_FEATURES = ["content_type", "main_intent"]
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_FEATURES),
])

def fit_and_score(train_idx, test_idx, label):
    logit = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
    X_tr, X_te = data.iloc[train_idx][FEATURE_COLS], data.iloc[test_idx][FEATURE_COLS]
    y_tr, y_te = data.iloc[train_idx]["is_page_one"], data.iloc[test_idx]["is_page_one"]
    logit.fit(X_tr, y_tr)
    proba = logit.predict_proba(X_te)[:, 1]
    print(f"{label:35} AUC = {roc_auc_score(y_te, proba):.3f}   AP = {average_precision_score(y_te, proba):.3f}   (n_test={len(y_te):,})")
    return roc_auc_score(y_te, proba), average_precision_score(y_te, proba)

# BEFORE -- ungrounded random row-level split (the dishonest version)
rand_train_idx, rand_test_idx = train_test_split(
    np.arange(len(data)), test_size=0.25, random_state=42, stratify=data["is_page_one"])
before_auc, before_ap = fit_and_score(rand_train_idx, rand_test_idx, "BEFORE -- random row split (dishonest)")

# AFTER -- grouped by client (the honest version, same as w05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
grp_train_idx, grp_test_idx = next(gss.split(data, data["is_page_one"], data["client_hash_id"]))
after_auc, after_ap = fit_and_score(grp_train_idx, grp_test_idx, "AFTER -- grouped by client (honest, w05)")

print(f"\nGap (random - grouped): AUC {before_auc - after_auc:+.3f}   AP {before_ap - after_ap:+.3f}")
print("A positive gap here means the random split was flattering the model with client-memorization -- exactly the illusion a grouped split is designed to catch.")


BEFORE -- random row split (dishonest) AUC = 0.654   AP = 0.656   (n_test=13,473)
AFTER -- grouped by client (honest, w05) AUC = 0.561   AP = 0.589   (n_test=4,049)

Gap (random - grouped): AUC +0.093   AP +0.068
A positive gap here means the random split was flattering the model with client-memorization -- exactly the illusion a grouped split is designed to catch.


**Reading the gap:** the random split scores meaningfully higher than the grouped split -- AUC 0.654 vs 0.561 (+0.093), Average Precision 0.656 vs 0.589 (+0.068). This gap **is** the finding: roughly 9 points of AUC that a random split would have reported as "model skill" was actually the model partially memorizing which client a page belonged to, not a genuine content-signal pattern. The honest, defensible number for this model is **AUC 0.561 / AP 0.589** (the grouped-split result) -- not the more flattering 0.654 / 0.656 a careless random split would have shown. This is a real, moderate illusion-gap, not a negligible one, and it's the single clearest argument in this whole capstone for why the grouped-by-client split was the right choice from the start rather than a formality.

## 3. Leakage audit

The same hunt from w03/w03b, repeated here on the **final** feature set actually used in w05's model (`word_count`, `content_age_days`, `search_volume`, `competition`, `backlinks`, `content_type`, `main_intent`) -- confirming nothing leaky crept back in between w03b and the finished model.

In [5]:
from sklearn.metrics import roc_auc_score as auc_score

def evaluate_feature_set(cols, cat_cols, label_name):
    pre_local = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
         [c for c in cols if c not in cat_cols]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
         cat_cols),
    ])
    pipe = Pipeline([("pre", pre_local), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
    X_tr, X_te = data.iloc[grp_train_idx][cols], data.iloc[grp_test_idx][cols]
    y_tr, y_te = data.iloc[grp_train_idx]["is_page_one"], data.iloc[grp_test_idx]["is_page_one"]
    pipe.fit(X_tr, y_tr)
    score = auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
    print(f"{label_name:45} AUC = {score:.3f}")
    return score

honest_auc = evaluate_feature_set(FEATURE_COLS, CAT_FEATURES, "Final feature set (honest, w05)")

# Re-run the confession test one more time, ON the exact final pipeline/split used for the paper's numbers
label_auc = evaluate_feature_set(FEATURE_COLS + ["avg_position_win"], CAT_FEATURES, "+ avg_position_win (the label itself)")
print(f"\nConfession gap: {label_auc - honest_auc:+.3f}  -- expect a large jump; if it's small, the harness itself may be broken, not the features.")


Final feature set (honest, w05)               AUC = 0.561
+ avg_position_win (the label itself)         AUC = 0.997

Confession gap: +0.436  -- expect a large jump; if it's small, the harness itself may be broken, not the features.


**Verdict:** the final feature set used for every reported w05 number contains none of the columns confirmed leaky in w03b (`avg_position_win`, `clicks_win`, the four query-mix columns). Re-running the confession test here, on the exact pipeline and split that produced the paper's headline numbers (not just on the earlier exploratory feature set), is the point -- it confirms the leakage check still holds at the final stage, not just when it was first tested three weeks ago.

## 4. Claim rewrite

**My boldest sentence, as originally written (w05, Section 4):** *"Logistic Regression wins on both ROC AUC (0.561) and Average Precision (0.589), ahead of both the Rule baseline (0.479 / 0.523) and Random Forest (0.517 / 0.545)."*

**Rewritten in safe, evidence-matched language:** *"On a single held-out split of 8 clients (4,049 pages), Logistic Regression showed the highest ROC AUC (0.561) and Average Precision (0.589) among the three methods compared, ahead of the rule baseline and Random Forest on this slice. Given the small number of held-out clients, this should be read as an observed, directional result from one split rather than a confirmed, stable ranking -- the before/after check in Section 2 and repeated-split testing would be needed before treating 'Logistic Regression beats the rule' as a settled finding rather than a promising first read."*

**What changed and why:** the original sentence used "wins," stated as if settled -- the rewrite adds the sample-size context (8 clients is a small held-out group), swaps "wins" for "showed the highest," and explicitly names what additional evidence (repeated splits) would be needed before the claim graduates from "directional, one split" to something sturdier. This follows the claim ladder directly: a single validated comparison supports "X showed higher performance than Y in this test," not "X beats Y," full stop.

In [6]:
# Reasoning-only -- the rewrite above is the deliverable; no additional query needed.
print("Original claim: causal-sounding 'wins', no sample-size context.")
print("Rewritten claim: directional, ties the number to n=8 clients, names what more evidence would be needed.")


Original claim: causal-sounding 'wins', no sample-size context.
Rewritten claim: directional, ties the number to n=8 clients, names what more evidence would be needed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.